# CineData Analytics — Landing → Bronze
**Camada Bronze:** dados brutos, sem alteração estrutural ou de conteúdo.

- Lê os 5 CSVs do Volume (todas as colunas como `string`, para não perder nada na ingestão).
- Adiciona `ingestion_datetime` (timestamp exato da inserção).
- Grava em Delta, modo **append** (histórico de ingestões preservado; a deduplicação é feita na Silver).
- Ingere a cotação do dólar (PTAX/Banco Central) em `bronze.tb_cotacao_dolar`.

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import requests

# ---------- Parâmetros (widgets) ----------
dbutils.widgets.text("volume_path", "/Volumes/workspace/default/cinedata_inputs", "Caminho do Volume com os CSVs")
dbutils.widgets.text("data_inicio", "", "Data início cotação (MM-DD-AAAA) - vazio = hoje-7")
dbutils.widgets.text("data_fim", "", "Data fim cotação (MM-DD-AAAA) - vazio = hoje")

VOLUME_PATH = dbutils.widgets.get("volume_path").rstrip("/")
FMT_API = "%m-%d-%Y"   # formato exigido pela API do BCB

hoje = datetime.now()
data_fim = datetime.strptime(dbutils.widgets.get("data_fim"), FMT_API) if dbutils.widgets.get("data_fim") else hoje
data_inicio = datetime.strptime(dbutils.widgets.get("data_inicio"), FMT_API) if dbutils.widgets.get("data_inicio") else data_fim - timedelta(days=7)
print(VOLUME_PATH, data_inicio.date(), data_fim.date())

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

## 1. Tabelas Bronze a partir dos CSVs

In [ ]:
import os

# (prefixo do arquivo, tabela Bronze, multiLine)
# multiLine=True só onde há texto livre com quebras de linha entre aspas (sinopses, comentários).
# Em metrics e credits NÃO há quebras legítimas e há aspas desbalanceadas (column shift de origem):
# com multiLine=True o parser engoliria várias linhas em um único registro e PERDERIA filmes.
ARQUIVOS = [
    ("movies_info",       "bronze.tb_movies_info",       True),
    ("movies_financials", "bronze.tb_movies_financials", True),
    ("movies_metrics",    "bronze.tb_movies_metrics",    False),
    ("credits_and_tags",  "bronze.tb_credits_and_tags",  False),
    ("movies_reviews",    "bronze.tb_movies_reviews",    True),
]

def resolve_arquivo(prefixo):
    """Localiza o CSV pelo prefixo (o nome do info vem como TMDB_IMDB ou IMDB_TMDB conforme a fonte)."""
    achados = [f for f in os.listdir(VOLUME_PATH) if f.lower().startswith(prefixo) and f.lower().endswith(".csv")]
    if len(achados) != 1:
        raise FileNotFoundError(f"Esperava 1 arquivo '{prefixo}*.csv' em {VOLUME_PATH}, achei: {achados}")
    return achados[0]

def grava_bronze(df, tabela):
    """Adiciona ingestion_datetime e grava em Delta (append)."""
    (df.withColumn("ingestion_datetime", F.current_timestamp())
       .write.format("delta").mode("append")
       .option("mergeSchema", "true")
       .saveAsTable(tabela))

for prefixo, tabela, multiline in ARQUIVOS:
    arquivo = resolve_arquivo(prefixo)
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)   # tudo string: nenhuma alteração de conteúdo na Bronze
          .option("multiLine", multiline)
          .option("quote", '"').option("escape", '"')
          .option("encoding", "UTF-8")
          .csv(f"{VOLUME_PATH}/{arquivo}"))
    grava_bronze(df, tabela)
    print(f"{tabela} <- {arquivo}")

## 2. Ingestão via API — cotação do dólar (PTAX, Banco Central)
A API não retorna cotação em fins de semana/feriados, por isso a janela padrão é de 7 dias corridos.
O preenchimento dos dias sem cotação (forward fill) acontece na Silver.

In [ ]:
URL = ("https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
       "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
       "?@dataInicial='{ini}'&@dataFinalCotacao='{fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json")

url = URL.format(ini=data_inicio.strftime(FMT_API), fim=data_fim.strftime(FMT_API))
resp = requests.get(url, timeout=60)
resp.raise_for_status()
registros = resp.json().get("value", [])
print(f"{len(registros)} cotações retornadas")

if registros:
    # Mantém os valores como string (camada Bronze = dado cru)
    linhas = [(r["dataHoraCotacao"], str(r["cotacaoCompra"])) for r in registros]
    df_cot = spark.createDataFrame(linhas, "dataHoraCotacao string, cotacaoCompra string")
    grava_bronze(df_cot, "bronze.tb_cotacao_dolar")
else:
    print("API sem retorno para o período: nada gravado.")

In [ ]:
display(spark.table("bronze.tb_cotacao_dolar").orderBy(F.desc("ingestion_datetime")).limit(10))